In [126]:
!pip -q install geopandas folium branca rapidfuzz openpyxl requests shapely

In [127]:
import io
import re
import json
import requests
import pandas as pd
import geopandas as gpd

from rapidfuzz import process, fuzz

import folium
from folium import GeoJson
from folium.features import GeoJsonTooltip
from branca.colormap import LinearColormap, StepColormap

from shapely.ops import unary_union


In [128]:
GEOJSON_PATH = "INDIAN_SUB_DISTRICTS.geojson"
india_gdf = gpd.read_file(GEOJSON_PATH)

print(india_gdf.head())

   OBJECTID stcode11 dtcode11 sdtcode11  Shape_Length    Shape_Area  \
0      4808       35      638     05916  5.206834e+04  1.295483e+08   
1      4809       35      638     05917  4.928417e+05  4.692465e+08   
2      4810       35      638     05918  3.536977e+05  1.120545e+09   
3      4811       35      639     05919  1.127189e+06  1.428804e+09   
4      4812       35      639     05920  5.380326e+05  8.030010e+08   

              stname                   dtname        sdtname  Subdt_LGD  \
0  ANDAMAN & NICOBAR                 Nicobars    Car Nicobar     5916.0   
1  ANDAMAN & NICOBAR                 Nicobars       Nancowry     5917.0   
2  ANDAMAN & NICOBAR                 Nicobars  Great Nicobar     5918.0   
3  ANDAMAN & NICOBAR  North  & Middle Andaman       Diglipur     5919.0   
4  ANDAMAN & NICOBAR  North  & Middle Andaman     Mayabunder     5920.0   

   Dist_LGD  State_LGD                                           geometry  
0     603.0         35  POLYGON ((92.77359 9.2

In [129]:
state_name_col = "stname"

maharashtra_gdf = india_gdf[
    india_gdf[state_name_col].str.lower() == "maharashtra"
]

maharashtra_gdf = maharashtra_gdf.reset_index(drop=True)


print(maharashtra_gdf.columns.tolist())
print(maharashtra_gdf.head())
print(f"Number of sub-districts in Maharashtra: {len(maharashtra_gdf)}")
maharashtra_gdf.tail()

['OBJECTID', 'stcode11', 'dtcode11', 'sdtcode11', 'Shape_Length', 'Shape_Area', 'stname', 'dtname', 'sdtname', 'Subdt_LGD', 'Dist_LGD', 'State_LGD', 'geometry']
   OBJECTID stcode11 dtcode11 sdtcode11   Shape_Length    Shape_Area  \
0      3969       27      497     03950  229977.366751  1.084474e+09   
1      3970       27      497     03951  206186.733326  1.498276e+09   
2      3971       27      497     03952  162472.061389  5.141909e+08   
3      3972       27      497     03953  195674.992974  1.405330e+09   
4      3973       27      497     03954  212746.249040  1.216660e+09   

        stname     dtname    sdtname  Subdt_LGD  Dist_LGD  State_LGD  \
0  MAHARASHTRA  Nandurbar  Akkalkuwa     3950.0     486.0         27   
1  MAHARASHTRA  Nandurbar     Akrani     3951.0     486.0         27   
2  MAHARASHTRA  Nandurbar     Talode     3952.0     486.0         27   
3  MAHARASHTRA  Nandurbar    Shahade     3953.0     486.0         27   
4  MAHARASHTRA  Nandurbar  Nandurbar     3954.

,OBJECTID,stcode11,dtcode11,sdtcode11,Shape_Length,Shape_Area,stname,dtname,sdtname,Subdt_LGD,Dist_LGD,State_LGD,geometry
354,4324,27,732,04163,279359.485875,1.184824e+09,MAHARASHTRA,Palghar,Palghar,4163.0,665.0,27,"POLYGON ((72.70018 19.85942, 72.6884 19.8611, ..."
355,4325,27,732,04164,141935.022250,6.303272e+08,MAHARASHTRA,Palghar,Vasai,4164.0,665.0,27,"POLYGON ((72.97548 19.55681, 72.96679 19.55857..."
356,5950,27,526,04252,184316.344630,1.260304e+09,MAHARASHTRA,Solapur,Mangalvedhe,4252.0,496.0,27,"POLYGON ((75.63305 17.47722, 75.62554 17.48786..."
357,5960,27,518,05927,205.007168,4.196569e+00,MAHARASHTRA,Mumbai Suburban,Mumbai Suburban,0.0,483.0,27,"POLYGON ((72.83468 19.0493, 72.83433 19.04937,..."
358,5961,27,518,05927,255.287603,1.301300e+01,MAHARASHTRA,Mumbai Suburban,Mumbai Suburban,0.0,483.0,27,"POLYGON ((72.83687 19.04898, 72.83585 19.0494,..."


In [130]:

mumbai_sub_gdf = maharashtra_gdf[maharashtra_gdf["sdtname"] == 'Mumbai Suburban']

print(mumbai_sub_gdf.head())
for col in mumbai_sub_gdf.columns:
    print(f"{col}: {mumbai_sub_gdf[col].dtype}")

# Dissolve all 3 rows into 1, unioning the geometries
mumbai_sub_merged = mumbai_sub_gdf.dissolve(
    by="sdtcode11",         # group by the subdist code (same for all 3)
    aggfunc={
        "Shape_Length": "sum",   # sum lengths
        "Shape_Area": "sum",     # sum areas
        # Keep first value for identifier columns (all same across rows)
        "OBJECTID": "first",
        "stcode11": "first",
        "dtcode11": "first",
        "stname": "first",
        "dtname": "first",
        "sdtname": "first",
        "Subdt_LGD": "first",
        "Dist_LGD": "first",
        "State_LGD": "first",
    }
).reset_index()

print(mumbai_sub_merged.shape)       # should be (1, ...)
print(mumbai_sub_merged["Shape_Area"].values)   # should be ~490M
print(mumbai_sub_merged.geometry)
print(mumbai_sub_merged.head())

     OBJECTID stcode11 dtcode11 sdtcode11   Shape_Length    Shape_Area  \
215      4184       27      518     05927  118971.158704  4.908704e+08   
357      5960       27      518     05927     205.007168  4.196569e+00   
358      5961       27      518     05927     255.287603  1.301300e+01   

          stname           dtname          sdtname  Subdt_LGD  Dist_LGD  \
215  MAHARASHTRA  Mumbai Suburban  Mumbai Suburban        0.0     483.0   
357  MAHARASHTRA  Mumbai Suburban  Mumbai Suburban        0.0     483.0   
358  MAHARASHTRA  Mumbai Suburban  Mumbai Suburban        0.0     483.0   

     State_LGD                                           geometry  
215         27  POLYGON ((72.83817 19.04825, 72.83912 19.04807...  
357         27  POLYGON ((72.83468 19.0493, 72.83433 19.04937,...  
358         27  POLYGON ((72.83687 19.04898, 72.83585 19.0494,...  
OBJECTID: int32
stcode11: str
dtcode11: str
sdtcode11: str
Shape_Length: float64
Shape_Area: float64
stname: str
dtname: str
sdtna

In [131]:
maharashtra_gdf = maharashtra_gdf[maharashtra_gdf["sdtname"] != 'Mumbai Suburban']
maharashtra_gdf = pd.concat([maharashtra_gdf, mumbai_sub_merged], ignore_index=True)

maharashtra_gdf = maharashtra_gdf.reset_index(drop=True)
maharashtra_gdf = maharashtra_gdf.sort_values(by="sdtcode11").reset_index(drop=True)
print(maharashtra_gdf.shape)
maharashtra_gdf.tail()
print("Total Area of Maharashtra sub-districts:", maharashtra_gdf["Shape_Area"].sum())

(357, 13)
Total Area of Maharashtra sub-districts: 347857031918.51825


In [132]:
sub_district_name_col = "sdtname"
census_data_path = "pop_area.csv"

census_df = pd.read_csv(census_data_path)

print(census_df.columns.tolist())
print(census_df.head())
print(f"Number of rows in census data: {len(census_df)}")

['State Code', 'District Code', 'Sub District Code', 'Administrative Unit', 'Name', 'Total/Rural/Urban', 'Number of Inhabited villages', 'Number of Uninhabited villages', 'Number of towns', 'Number of households', 'Total Population', 'Male Population', 'Female Population', 'Area(In sq. km)', 'Population per sq. km.']
   State Code District Code Sub District Code Administrative Unit  \
0         0.0           000             00000               INDIA   
1         0.0           000             00000               INDIA   
2         0.0           000             00000               INDIA   
3         1.0           000             00000               STATE   
4         1.0           000             00000               STATE   

                 Name Total/Rural/Urban  Number of Inhabited villages  \
0            INDIA @&             Total                      597608.0   
1             INDIA $             Rural                      597608.0   
2             INDIA $             Urban        

In [133]:
census_df = census_df[census_df["Total/Rural/Urban"].str.lower() == "total"]
maharashtra_census_df = census_df[census_df["State Code"] == 27]
maharashtra_census_df = maharashtra_census_df[maharashtra_census_df["Administrative Unit"].str.lower() == "sub-district"]

maharashtra_census_df = maharashtra_census_df.reset_index(drop=True)

# print(maharashtra_census_df.tail())
# print(f"Number of rows in maharashtra census data: {len(maharashtra_census_df)}")

# print(f"Total area of maharashtra sub-districts: {maharashtra_census_df['Area (in sq. km.)'].sum()}")

# print(f"Top 10 sub-districts by population:\n{maharashtra_census_df.sort_values(by='Total Population', ascending=False).head(10)}")
maharashtra_census_df.loc[maharashtra_census_df["District Code"] == "518", ["Name", "Sub District Code"]] = "Mumbai Suburban", "05927"
maharashtra_census_df.loc[maharashtra_census_df["District Code"] == "519", ["Name", "Sub District Code"]] = "Greater Mumbai", "05926"

maharashtra_census_df = maharashtra_census_df.sort_values(by="District Code").reset_index(drop=True)
print(maharashtra_census_df.head())
print(maharashtra_census_df.tail())
print(f"Number of rows in maharashtra census data: {len(maharashtra_census_df)}")



   State Code District Code Sub District Code Administrative Unit       Name  \
0        27.0           497             03950        SUB-DISTRICT  Akkalkuwa   
1        27.0           497             03951        SUB-DISTRICT     Akrani   
2        27.0           497             03952        SUB-DISTRICT     Talode   
3        27.0           497             03953        SUB-DISTRICT    Shahade   
4        27.0           497             03954        SUB-DISTRICT  Nandurbar   

  Total/Rural/Urban  Number of Inhabited villages  \
0             Total                         189.0   
1             Total                         162.0   
2             Total                          92.0   
3             Total                         182.0   
4             Total                         149.0   

   Number of Uninhabited villages  Number of towns  Number of households  \
0                             1.0              3.0               46429.0   
1                             0.0              1

In [134]:
# drop unnecessary columns from maharashtra_census_df

print(f"Columns in maharashtra_census_df before dropping: {maharashtra_census_df.columns.tolist()}")
maharashtra_census_df = maharashtra_census_df.drop(columns=[
    "State Code",
    "District Code",
    "Total/Rural/Urban",
    "Administrative Unit",
    "Number of Inhabited villages",
    "Number of Uninhabited villages",
    "Number of towns",
    "Number of households",
    "Male Population",
    "Female Population"
])

print(f"Columns in maharashtra_census_df after dropping: {maharashtra_census_df.columns.tolist()}")


Columns in maharashtra_census_df before dropping: ['State Code', 'District Code', 'Sub District Code', 'Administrative Unit', 'Name', 'Total/Rural/Urban', 'Number of Inhabited villages', 'Number of Uninhabited villages', 'Number of towns', 'Number of households', 'Total Population', 'Male Population', 'Female Population', 'Area(In sq. km)', 'Population per sq. km.']
Columns in maharashtra_census_df after dropping: ['Sub District Code', 'Name', 'Total Population', 'Area(In sq. km)', 'Population per sq. km.']


In [135]:
# Drop unnecessary columns from maharashtra_gdf

print(f"Columns in maharashtra_gdf before dropping: {maharashtra_gdf.columns.tolist()}")

maharashtra_gdf = maharashtra_gdf.drop(columns=[
    "OBJECTID",
    "stcode11",
    "dtcode11",
    "stname",
    "dtname",
    "stname",
    "dtname",
    "Subdt_LGD",
    "Dist_LGD",
    "State_LGD"
])

print(f"Columns in maharashtra_gdf after dropping: {maharashtra_gdf.columns.tolist()}")

Columns in maharashtra_gdf before dropping: ['OBJECTID', 'stcode11', 'dtcode11', 'sdtcode11', 'Shape_Length', 'Shape_Area', 'stname', 'dtname', 'sdtname', 'Subdt_LGD', 'Dist_LGD', 'State_LGD', 'geometry']
Columns in maharashtra_gdf after dropping: ['sdtcode11', 'Shape_Length', 'Shape_Area', 'sdtname', 'geometry']


In [136]:
print(f"Number of rows in maharashtra_gdf: {len(maharashtra_gdf)}")
print(f"Number of rows in maharashtra_census_df: {len(maharashtra_census_df)}")

# Compare side by side the maharashtra_gdf and maharashtra_census_df to see if they match

print("maharashtra_gdf:")
print(maharashtra_gdf.head())
print("\nmaharashtra_census_df:")
print(maharashtra_census_df.head())

# Merge



Number of rows in maharashtra_gdf: 357
Number of rows in maharashtra_census_df: 357
maharashtra_gdf:
  sdtcode11   Shape_Length    Shape_Area    sdtname  \
0     03950  229977.366751  1.084474e+09  Akkalkuwa   
1     03951  206186.733326  1.498276e+09     Akrani   
2     03952  162472.061389  5.141909e+08     Talode   
3     03953  195674.992974  1.405330e+09    Shahade   
4     03954  212746.249040  1.216660e+09  Nandurbar   

                                            geometry  
0  POLYGON ((74.12203 21.68172, 74.11657 21.68949...  
1  POLYGON ((74.43172 22.03022, 74.41519 22.02487...  
2  POLYGON ((74.34953 21.69638, 74.33601 21.69471...  
3  POLYGON ((74.50176 21.78149, 74.49435 21.77997...  
4  POLYGON ((74.49574 21.42568, 74.44332 21.44673...  

maharashtra_census_df:
  Sub District Code       Name  Total Population  Area(In sq. km)  \
0             03950  Akkalkuwa          245861.0           936.02   
1             03951     Akrani          195754.0          1282.31   
2      

In [137]:
merged_df = maharashtra_gdf.merge(maharashtra_census_df, left_on="sdtcode11", right_on="Sub District Code", how="outer", indicator=True)

print(f"Number of rows in merged_df: {len(merged_df)}")


merged_df.drop(columns=["_merge", "Name"], inplace=True)
merged_df = merged_df.sort_values(by="sdtcode11").reset_index(drop=True)
print(merged_df.head())

Number of rows in merged_df: 357
  sdtcode11   Shape_Length    Shape_Area    sdtname  \
0     03950  229977.366751  1.084474e+09  Akkalkuwa   
1     03951  206186.733326  1.498276e+09     Akrani   
2     03952  162472.061389  5.141909e+08     Talode   
3     03953  195674.992974  1.405330e+09    Shahade   
4     03954  212746.249040  1.216660e+09  Nandurbar   

                                            geometry Sub District Code  \
0  POLYGON ((74.12203 21.68172, 74.11657 21.68949...             03950   
1  POLYGON ((74.43172 22.03022, 74.41519 22.02487...             03951   
2  POLYGON ((74.34953 21.69638, 74.33601 21.69471...             03952   
3  POLYGON ((74.50176 21.78149, 74.49435 21.77997...             03953   
4  POLYGON ((74.49574 21.42568, 74.44332 21.44673...             03954   

   Total Population  Area(In sq. km)  Population per sq. km.  
0          245861.0           936.02              262.666396  
1          195754.0          1282.31              152.657314  
2 

In [ ]:
import numpy as np

# Ensure data is in WGS84 for folium
map_gdf = merged_df.to_crs(epsg=4326).copy()

density_col = "Population per sq. km."
map_gdf[density_col] = pd.to_numeric(map_gdf[density_col], errors="coerce")

# Log transform to spread out large disparities
density_log_col = "_density_log"
map_gdf[density_log_col] = np.log1p(map_gdf[density_col])

# Center the map on Maharashtra
minx, miny, maxx, maxy = map_gdf.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=7, tiles="cartodbpositron")

# Smooth color scale on log values
vmin = map_gdf[density_log_col].min()
vmax = map_gdf[density_log_col].max()
colormap = LinearColormap(
    colors=["#ffffcc", "#fed976", "#fd8d3c", "#f03b20", "#800026"],
    vmin=vmin,
    vmax=vmax,
    caption="Population density (per sq. km.)"
)

def style_function(feature):
    value = feature["properties"].get(density_log_col)
    return {
        "fillColor": colormap(value) if value is not None else "#808080",
        "color": "black",
        "weight": 0.5,
        "fillOpacity": 0.75,
    }

geojson = folium.GeoJson(
    map_gdf,
    name="Maharashtra Sub-districts",
    style_function=style_function,
    tooltip=GeoJsonTooltip(
        fields=["sdtname", "Total Population", "Area(In sq. km)", density_col],
        aliases=["Sub-district", "Population", "Area (sq. km)", "Density"],
        localize=True,
        sticky=False
    ),
)

geojson.add_to(m)
colormap.add_to(m)
folium.LayerControl().add_to(m)

m.save("maharashtra_population_density_map.html")